# Affiner le modèle de l'agent par QLoRA, comparer deux configurations

Le fil rouge de la formation : l'agent du service client de Biclou & Cie, qui lit le message d'un client, appelle les outils du magasin (la fiche d'une commande, le stock, les règles du magasin, un ticket, le transfert à un conseiller), puis répond selon la charte. Le TP précédent a relevé les erreurs de son modèle, Qwen3-1.7B, et en a tiré un jeu d'entraînement d'appels d'outils. Celui-ci affine Qwen3-1.7B par QLoRA sur la version de référence de ce jeu, celle que le même code a construite sur GPU, pour ne pas dépendre de votre exécution. Il compare deux configurations de LoRA, mesure les deux adaptateurs et le modèle de base sur le jeu d'évaluation figé, puis enregistre le meilleur avec ce qu'il faut pour refaire l'expérience : le chapitre suivant le branche dans l'agent.

1. Estimer la mémoire
2. Préparer l'entraînement
3. La configuration A, celle du cours
4. La configuration B, l'attention seule
5. Mesurer et choisir
6. Enregistrer, versionner, publier

Ce notebook demande un GPU : *Exécution > Modifier le type d'exécution > GPU T4*, sur lequel il tient en une heure à une heure et demie (une estimation), dont les trois quarts pour les deux entraînements. Sans GPU, il tourne quand même, en plus petit : Qwen3-0.6B en float32 au lieu de Qwen3-1.7B en 4 bits, entraîné sur 2 exemples au lieu du jeu entier (le code tourne, mais les courbes ne disent rien), et l'exercice 5 mesure, sur 6 demandes du jeu figé au lieu de 139, le modèle de base et, à la place des vôtres, un adaptateur de Qwen3-0.6B que ce notebook a entraîné sur GPU : trois quarts d'heure environ sur deux processeurs, comme ceux de Colab (une estimation).

## Données et modèle

`agent-sft-train.jsonl` et `agent-sft-validation.jsonl` : le jeu d'entraînement et celui de validation du TP précédent, dans leur version de référence, au format conversationnel de `trl` : une colonne `messages` (la consigne de l'agent, le message du client, chaque appel dans un message `assistant` suivi de son résultat dans un message `tool`, puis la réponse) et une colonne `tools`, les schémas des cinq outils en texte JSON. `eval.jsonl` : le jeu d'évaluation figé, 139 demandes avec les appels attendus (`appels_attendus`), leurs résultats (`resultats_outils`) et la réponse, sur lequel les chapitres suivants comparent les modèles, et sur lequel rien n'est entraîné. `outils.json` : les schémas des cinq outils. `agent-sft-lora-qwen3-0.6b/` : l'adaptateur de Qwen3-0.6B que mesure l'exercice 5 quand le notebook tourne sans GPU.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/lab-results/agent-sft-train.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/lab-results/agent-sft-validation.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/eval.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/outils.json
!wget -q -P agent-sft-lora-qwen3-0.6b https://github.com/shuuchuu/models/raw/refs/heads/main/agent-sft-lora-qwen3-0.6b/adapter_model.safetensors
!wget -q -P agent-sft-lora-qwen3-0.6b https://github.com/shuuchuu/models/raw/refs/heads/main/agent-sft-lora-qwen3-0.6b/adapter_config.json
%pip install -q peft trl bitsandbytes
%pip uninstall -q -y torchao  # la version de Colab est trop ancienne pour peft

In [ ]:
import copy
import gc
import json
import os
import re
import time

import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from transformers import (
  AutoModelForCausalLM,
  AutoTokenizer,
  BitsAndBytesConfig,
  DynamicCache,
)
from trl import SFTConfig, SFTTrainer

GPU = torch.cuda.is_available()
# bfloat16 à partir des GPU Ampere (A100, L4, H100) ; le T4 calcule en float16.
BF16 = GPU and torch.cuda.get_device_capability()[0] >= 8
# Sur CPU, float32 : les demi-précisions y sont très lentes.
DTYPE = (torch.bfloat16 if BF16 else torch.float16) if GPU else torch.float32
DEVICE = "cuda" if GPU else "cpu"
MODELE = "Qwen/Qwen3-1.7B" if GPU else "Qwen/Qwen3-0.6B"
# Exemples par pas et pas accumulés par mise à jour : 8 exemples sur GPU, 1 sans. Un
# exemple fait près de 2 000 tokens : 2 par pas sur un T4.
LOT = 2 if GPU else 1
ACCUMULATION = 4 if GPU else 1
print(MODELE, "sur GPU" if GPU else "sans GPU")

tokenizer = AutoTokenizer.from_pretrained(MODELE)
entrainement = load_dataset("json", data_files="agent-sft-train.jsonl", split="train")
validation = load_dataset(
  "json", data_files="agent-sft-validation.jsonl", split="train"
)
eval_fige = pd.read_json("eval.jsonl", lines=True)
with open("outils.json", encoding="utf-8") as f:
  OUTILS = json.load(f)
# Sans GPU : 2 exemples d'entraînement, 1 de validation, et 6 demandes du jeu figé,
# 2 sans appel attendu, 2 à un appel, 2 à deux appels.
nombre_appels = eval_fige["appels_attendus"].str.len()
if not GPU:
  entrainement = entrainement.select(range(2))
  validation = validation.select(range(1))
eval_part = eval_fige if GPU else eval_fige.groupby(nombre_appels).head(2)
print(len(entrainement), "exemples d'entraînement,", len(validation), "de validation,")
print(len(eval_part), "demandes du jeu figé à mesurer")


def memoire_go(modele) -> float:
  """La mémoire qu'occupe le modèle, en Go : celle du GPU, sinon celle de ses poids."""
  if GPU:
    return torch.cuda.memory_allocated() / 1e9
  return modele.get_memory_footprint() / 1e9


def pic_go() -> float:
  """Le pic de mémoire GPU depuis le dernier libere(), en Go (0 sans GPU)."""
  return torch.cuda.max_memory_allocated() / 1e9 if GPU else 0.0


def libere() -> None:
  """Rend au GPU la mémoire des objets supprimés par `del`, et remet le pic à zéro."""
  gc.collect()
  if GPU:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def taille_mo(dossier: str) -> float:
  """La taille des fichiers d'un dossier, en Mo."""
  return (
    sum(os.path.getsize(os.path.join(dossier, f)) for f in os.listdir(dossier)) / 1e6
  )

## La mesure

Celle du TP précédent, telle quelle : `CONSIGNES_AGENT`, la consigne de l'agent, celle de chaque exemple ; `lit_appels(texte)`, les appels d'une sortie du modèle (`None` si un bloc est illisible) ; `valide(appel)`, les problèmes d'un appel face à son schéma (`SCHEMAS`) ; `exacts(appel)`, ses arguments sans les arguments libres (`question` et `resume`) ; `erreur_de(attendus, trajet)`, la classe d'une sortie, de `format` à `correct` ; `prefixe` et `genere_tout`, qui génèrent sans échantillonnage, par lots de 8 sur GPU, une à une sans GPU, à partir du cache du message `system` et des schémas, calculé une fois : sans lui, chaque génération relirait leurs 1 400 tokens environ.

Elle s'applique ici étape par étape. Pour chaque demande du jeu figé, `etapes(demande)` reconstruit l'historique correct à partir des appels attendus et de leurs résultats : le modèle doit produire le premier appel après le message du client, le deuxième après le premier et son résultat, et, après le dernier résultat, répondre sans appeler d'outil. Chaque étape est ainsi jugée sur le bon historique, même quand le modèle s'est trompé à la précédente, et sans exécuter les outils. `note(demandes, textes)` classe chaque étape (`erreur_de`), puis chaque demande par sa pire étape, en quatre taux qui s'emboîtent : `format_lisible` (toutes ses sorties sont lisibles), `bons_outils` (et chaque étape appelle l'outil attendu, ou aucun), `arguments_valides` (et chaque appel passe `valide`), `correct` (et les arguments sont les attendus, arguments libres à part) ; `erreur` est la classe de sa première étape fausse, et `etape` son rang. `mesure(modele)` génère toutes les étapes des demandes d'`eval_part` et rend ces taux, avec le temps par demande, et le détail par demande.

La réponse finale n'est jugée ici que sur un point : aucun appel. L'agent de bout en bout, avec ses propres appels, les vrais outils et la qualité de ses réponses, c'est l'évaluation du dernier chapitre.

In [ ]:
CONSIGNES_AGENT = entrainement[0]["messages"][0]["content"]
SCHEMAS = {outil["function"]["name"]: outil["function"] for outil in OUTILS}
ARGUMENTS_LIBRES = {"question", "resume"}
BLOC_APPEL = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)


def lit_appels(texte: str) -> list[dict] | None:
  """Les appels d'outil d'une sortie du modèle, None si un bloc est illisible."""
  appels = []
  for bloc in BLOC_APPEL.findall(texte):
    try:
      appel = json.loads(bloc)
      arguments = appel.get("arguments", {})
      if isinstance(arguments, str):
        arguments = json.loads(arguments)
    except (json.JSONDecodeError, AttributeError):
      return None
    if not isinstance(appel.get("name"), str):
      return None
    appels.append(dict(name=appel["name"], arguments=arguments))
  if texte.count("<tool_call>") != len(appels):  # un bloc pas refermé
    return None
  return appels


def valide(appel: dict) -> list[str]:
  """Les problèmes d'un appel face au schéma de son outil, [] s'il est valide."""
  schema = SCHEMAS.get(appel["name"])
  if schema is None:
    return [f"outil inconnu : {appel['name']}"]
  arguments = appel["arguments"]
  if not isinstance(arguments, dict):
    return ["les arguments ne sont pas un objet"]
  proprietes = schema["parameters"]["properties"]
  problemes = [
    f"{a} manquant" for a in schema["parameters"]["required"] if a not in arguments
  ]
  for a, valeur in arguments.items():
    if a not in proprietes:
      problemes.append(f"{a} inconnu")
    elif not isinstance(valeur, str):
      problemes.append(f"{a} n'est pas une chaîne")
    elif valeur not in proprietes[a].get("enum", [valeur]):
      problemes.append(f"{a} hors de la liste : {valeur}")
  numero = arguments.get("numero")
  if isinstance(numero, str) and not re.fullmatch(r"CMD-\d{6}", numero):
    problemes.append(f"numero mal formé : {numero}")
  return problemes


def exacts(appel: dict) -> dict:
  return {a: v for a, v in appel["arguments"].items() if a not in ARGUMENTS_LIBRES}


def erreur_de(attendus: list[dict], trajet: dict) -> str:
  obtenus = trajet["appels"]
  noms_attendus = [a["name"] for a in attendus]
  noms_obtenus = [a["name"] for a in obtenus]
  if not trajet["format_lisible"]:
    return "format"
  if noms_obtenus != noms_attendus:
    if noms_obtenus == noms_attendus[: len(noms_obtenus)]:
      return "appel_manquant"
    if noms_attendus == noms_obtenus[: len(noms_attendus)]:
      return "appel_inutile"
    return "mauvais_outil"
  if any(valide(a) for a in obtenus):
    return "arguments_invalides"
  if any(exacts(o) != exacts(a) for o, a in zip(obtenus, attendus)):
    return "arguments_faux"
  return "correct"


def appel_chat(appel: dict) -> dict:
  """Un appel au format des conversations : ses arguments en texte JSON."""
  arguments = json.dumps(appel["arguments"], ensure_ascii=False)
  return dict(type="function", function=dict(name=appel["name"], arguments=arguments))


def prefixe(modele, tok, systeme: str, outils: list | None = None):
  """Sans GPU, le cache du début commun des prompts (le message system et les
  outils), calculé une fois : chaque génération repart d'une copie, comme le fait un
  serveur d'inférence. Sur GPU, None : les lots vont assez vite sans."""
  if GPU:
    return None
  debut = tok.apply_chat_template(
    [dict(role="system", content=systeme)], tools=outils, tokenize=False
  )
  cache = DynamicCache(config=modele.config)
  with torch.no_grad():
    modele(**tok(debut, return_tensors="pt").to(DEVICE), past_key_values=cache)
  return cache


def genere_tout(
  modele, tok, conversations, outils=None, cache=None, max_tokens=96, **options
):
  """La suite de chaque conversation : par lots de 8 sur GPU, une à une sans GPU, à
  partir d'une copie de `cache`. Sans échantillonnage, sauf `options`."""
  options = options or dict(do_sample=False, temperature=None, top_p=None, top_k=None)
  prompts = [
    tok.apply_chat_template(
      c, tools=outils, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    for c in conversations
  ]
  taille_lot = 8 if GPU else 1
  textes = []
  for debut in range(0, len(prompts), taille_lot):
    entrees = tok(
      prompts[debut : debut + taille_lot], return_tensors="pt", padding=True
    )
    copie = copy.deepcopy(cache) if cache is not None else None
    with torch.no_grad():
      sorties = modele.generate(
        **entrees.to(DEVICE),
        past_key_values=copie,
        max_new_tokens=max_tokens,
        **options,
      )
    textes += tok.batch_decode(
      sorties[:, entrees.input_ids.shape[1] :], skip_special_tokens=True
    )
  return textes


def etapes(demande) -> list[tuple[list[dict], list[dict]]]:
  """Les étapes d'une demande : pour chaque appel attendu, la conversation qui le
  précède et cet appel ; puis la conversation entière, où aucun appel n'est attendu."""
  conversation = [
    dict(role="system", content=CONSIGNES_AGENT),
    dict(role="user", content=demande["message"]),
  ]
  liste = []
  for appel, resultat in zip(demande["appels_attendus"], demande["resultats_outils"]):
    liste.append((conversation, [appel]))
    conversation = conversation + [
      dict(role="assistant", content="", tool_calls=[appel_chat(appel)]),
      dict(role="tool", content=json.dumps(resultat, ensure_ascii=False)),
    ]
  liste.append((conversation, []))
  return liste


# Le niveau qu'atteint une étape : chaque taux compte les demandes dont toutes les
# étapes le dépassent.
NIVEAUX = {
  "format": 0,
  "appel_manquant": 1,
  "appel_inutile": 1,
  "mauvais_outil": 1,
  "arguments_invalides": 2,
  "arguments_faux": 3,
  "correct": 4,
}
TAUX = ["format_lisible", "bons_outils", "arguments_valides", "correct"]


def note(demandes: pd.DataFrame, textes: list[str]) -> pd.DataFrame:
  """Le détail par demande, des sorties du modèle à chaque étape, dans l'ordre."""
  textes = iter(textes)
  lignes = []
  for demande in demandes.to_dict("records"):
    sorties, erreurs = [], []
    for _, attendus in etapes(demande):
      sorties.append(next(textes))
      appels = lit_appels(sorties[-1])
      trajet = dict(appels=appels or [], format_lisible=appels is not None)
      erreurs.append(erreur_de(attendus, trajet))
    niveau = min(NIVEAUX[e] for e in erreurs)
    fausses = [i for i, e in enumerate(erreurs) if e != "correct"]
    lignes.append(
      dict(
        id=demande["id"],
        appels_attendus=len(demande["appels_attendus"]),
        **{nom: niveau > i for i, nom in enumerate(TAUX)},
        erreur=erreurs[fausses[0]] if fausses else "correct",
        etape=fausses[0] if fausses else None,
        sorties=sorties,
      )
    )
  return pd.DataFrame(lignes)


def mesure(modele) -> tuple[pd.Series, pd.DataFrame]:
  """Les taux de `modele` sur eval_part, avec le temps par demande, et le détail."""
  debut = time.time()
  demandes = eval_part.to_dict("records")
  conversations = [c for d in demandes for c, _ in etapes(d)]
  cache = prefixe(modele, tokenizer, CONSIGNES_AGENT, OUTILS)
  textes = genere_tout(modele, tokenizer, conversations, outils=OUTILS, cache=cache)
  detail = note(eval_part, textes)
  taux = detail[TAUX].mean()
  taux["s_par_demande"] = (time.time() - debut) / len(eval_part)
  return taux.round(2), detail


# Les étapes d'une demande à deux appels.
for conversation, attendus in etapes(eval_fige[nombre_appels == 2].iloc[0]):
  print(
    len(conversation), "messages ->", [a["name"] for a in attendus] or "aucun appel"
  )
print(sum(len(etapes(d)) for d in eval_fige.to_dict("records")), "étapes en tout")


# La mesure des sorties attendues elles-mêmes, chaque appel puis la réponse de
# référence : tout est correct.
def attendue(appel: dict) -> str:
  return f"<tool_call>\n{json.dumps(appel, ensure_ascii=False)}\n</tool_call>"


textes = [
  texte
  for d in eval_fige.to_dict("records")
  for texte in [*map(attendue, d["appels_attendus"]), d["reponse"]]
]
print(note(eval_fige, textes)[TAUX].mean().to_dict())

## 1. Estimer la mémoire

La cellule ci-dessous charge le modèle de base, `base`, en NF4 avec double quantification (sans GPU, en float32), puis donne les deux configurations à comparer et les réglages d'entraînement qu'elles partagent, `REFERENCE` : `lora_a`, celle du cours (rang 16, toutes les couches linéaires), et `lora_b`, la même sur les quatre projections de l'attention seules (`q_proj`, `k_proj`, `v_proj`, `o_proj`).

Avant d'entraîner, estimez ce que l'entraînement demandera au GPU. Écrivez `parametres_lora(lora)`, le nombre de paramètres qu'ajoute une configuration : pour chaque couche linéaire de `base` qu'elle cible (pour `"all-linear"`, toutes sauf `lm_head` ; sinon, celles dont le nom finit par l'un de ses `target_modules`), une matrice $A$ de `r` × `in_features` et une matrice $B$ de `out_features` × `r`. Rangez ceux des deux configurations dans `parametres`. Puis `estimation_go(lora)`, l'estimation en Go de la mémoire de l'entraînement, en trois parts :

- les poids de `base` (`memoire_go(base)`), gelés ;
- les paramètres LoRA, chacun avec son gradient et deux états d'AdamW : sur GPU, 2 + 2 + 2 octets (`trl` passe en bfloat16 les adaptateurs d'un modèle quantifié, et `paged_adamw_8bit` garde ses états sur 8 bits), sans GPU 4 + 4 + 8 (tout en float32) ;
- les activations : avec le *gradient checkpointing*, chaque couche ne garde que son entrée, `LOT` × `max_length` × `hidden_size` valeurs de `DTYPE`, et recalcule le reste pendant la rétropropagation.

Comparez les deux configurations : combien de paramètres entraînables, quelle part du modèle, combien de Go ? Qu'est-ce qui domine ? Et l'affinage complet du même modèle, à 16 octets par paramètre (les poids, leurs gradients et les états d'Adam), tiendrait-il sur les 15 Go d'un T4 ? L'exercice 3 mesure le pic réel. Sans GPU, il n'y a pas de *gradient checkpointing* ni de pic à mesurer (`pic_go()` vaut 0) : l'estimation n'y a rien à quoi se comparer.

**Documentation utile**

- [`Module.named_modules`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.named_modules)
- [`LoraConfig`](https://huggingface.co/docs/peft/package_reference/lora#peft.LoraConfig)
- [`num_parameters`](https://huggingface.co/docs/transformers/main_classes/model#transformers.modeling_utils.ModuleUtilsMixin.num_parameters)

In [ ]:
NF4 = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_use_double_quant=True,
  bnb_4bit_compute_dtype=DTYPE,
)
# Sans GPU, en float32 : bitsandbytes est lent sur processeur.
base = AutoModelForCausalLM.from_pretrained(
  MODELE, dtype=DTYPE, device_map=DEVICE, quantization_config=NF4 if GPU else None
)
print(f"Modèle de base : {memoire_go(base):.2f} Go")

# A : la configuration du cours, sur toutes les couches linéaires.
lora_a = LoraConfig(
  r=16,
  lora_alpha=32,
  lora_dropout=0.05,
  target_modules="all-linear",
  task_type="CAUSAL_LM",
)
# B : un seul changement, les modules ciblés, les projections de l'attention.
lora_b = LoraConfig(
  r=16,
  lora_alpha=32,
  lora_dropout=0.05,
  target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
  task_type="CAUSAL_LM",
)
REFERENCE = dict(
  per_device_train_batch_size=LOT,
  per_device_eval_batch_size=LOT,
  gradient_accumulation_steps=ACCUMULATION,
  # Une époque : le jeu compte des centaines d'exemples de près de 2 000 tokens.
  num_train_epochs=1,
  learning_rate=2e-4,
  lr_scheduler_type="cosine",
  warmup_steps=0.1,
  optim="paged_adamw_8bit" if GPU else "adamw_torch",
  # Économise la mémoire du GPU au prix d'un passage avant de plus ; inutile sans GPU.
  gradient_checkpointing=GPU,
  bf16=BF16,
  fp16=GPU and not BF16,
  max_length=2048,
)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `base.named_modules()` : des paires (nom, module), le nom complet, comme `model.layers.0.self_attn.q_proj` ; `nom.split(".")[-1]`, son dernier mot ; `isinstance(module, torch.nn.Linear)` : une couche linéaire (celles de `bitsandbytes`, en 4 bits, en sont aussi)
- `lora.target_modules` : la chaîne `"all-linear"`, ou l'ensemble des noms ciblés ; `lora.r`, le rang ; `module.in_features`, `module.out_features`
- `base.config.num_hidden_layers`, `base.config.hidden_size` ; `DTYPE.itemsize`, les octets d'une valeur ; `base.num_parameters()`, les paramètres du modèle
- Exemples à vérifier : sur Qwen3-1.7B, 17 432 576 paramètres pour `lora_a` et 6 422 528 pour `lora_b` ; sur Qwen3-0.6B, 10 092 544 et 4 587 520

```python
def parametres_lora(lora: LoraConfig) -> int:
  total = 0
  for nom, module in base.named_modules():
    if not isinstance(module, torch.nn.Linear) or nom.endswith("lm_head"):
      continue
    if lora.target_modules == "all-linear" or ... in lora.target_modules:
      total += lora.r * (...)
  return total


def estimation_go(lora: LoraConfig) -> pd.Series:
  octets = 2 + 2 + 2 if GPU else 4 + 4 + 8
  config = base.config
  reprises = (
    config.num_hidden_layers * LOT * REFERENCE["max_length"] * ... * DTYPE.itemsize
  )
  estimation = pd.Series(
    {
      "poids": memoire_go(base),
      "lora": parametres_lora(lora) * octets / 1e9,
      "activations": ...,
    }
  )
  estimation["total"] = estimation.sum()
  return estimation.round(2)


parametres = {"config-a": parametres_lora(lora_a), "config-b": parametres_lora(lora_b)}
for nom, n in parametres.items():
  print(f"{nom} : {n} paramètres entraînables, {n / base.num_parameters():.2%}")
estimations = pd.DataFrame(
  {"config-a": estimation_go(lora_a), "config-b": estimation_go(lora_b)}
)
print(estimations)
print(f"Affinage complet : {base.num_parameters() * 16 / 1e9:.0f} Go")
```

### Solution

In [ ]:
def parametres_lora(lora: LoraConfig) -> int:
  total = 0
  for nom, module in base.named_modules():
    if not isinstance(module, torch.nn.Linear) or nom.endswith("lm_head"):
      continue
    if lora.target_modules == "all-linear" or nom.split(".")[-1] in lora.target_modules:
      total += lora.r * (module.in_features + module.out_features)
  return total


def estimation_go(lora: LoraConfig) -> pd.Series:
  octets = 2 + 2 + 2 if GPU else 4 + 4 + 8
  config = base.config
  reprises = (
    config.num_hidden_layers
    * LOT
    * REFERENCE["max_length"]
    * config.hidden_size
    * DTYPE.itemsize
  )
  estimation = pd.Series(
    {
      "poids": memoire_go(base),
      "lora": parametres_lora(lora) * octets / 1e9,
      "activations": reprises / 1e9,
    }
  )
  estimation["total"] = estimation.sum()
  return estimation.round(2)


parametres = {"config-a": parametres_lora(lora_a), "config-b": parametres_lora(lora_b)}
for nom, n in parametres.items():
  print(f"{nom} : {n} paramètres entraînables, {n / base.num_parameters():.2%}")
estimations = pd.DataFrame(
  {"config-a": estimation_go(lora_a), "config-b": estimation_go(lora_b)}
)
print(estimations)
print(f"Affinage complet : {base.num_parameters() * 16 / 1e9:.0f} Go")

# Les poids gelés dominent, puis les activations ; LoRA ne pèse presque rien, 1 % du
# modèle pour A, près de trois fois moins pour B, une centaine de Mo au plus avec
# gradients et optimiseur. L'affinage complet de Qwen3-1.7B demanderait près de 28 Go :
# deux T4, avant même les activations.

## 2. Préparer l'entraînement

Écrivez `prepare(nom, lora, **changements)`, qui renvoie un `SFTTrainer` prêt à entraîner l'adaptateur `lora` sur `base` pendant une époque d'`entrainement`, avec les réglages de `REFERENCE` (que `changements` remplacent), et :

- la perte sur les seuls tours de l'assistant (`assistant_only_loss`) : le modèle apprend à appeler les outils et à répondre, pas à écrire les messages des clients ni les résultats des outils ;
- les journaux dans TensorBoard, sous `sft/<nom>`, avec la perte d'entraînement à chaque pas ;
- une évaluation sur `validation` à chaque dixième de l'entraînement, et un checkpoint à chacune ;
- deux checkpoints gardés au plus, dont toujours le meilleur selon la perte de validation, rechargé à la fin.

`trl` 1.18 lit la colonne `tools` telle qu'elle est enregistrée, en texte JSON : il la décode lui-même (`json.loads`) avant de la passer au *chat template* avec chaque conversation, et il n'y a rien à convertir.

Préparez la configuration A, `trainer = prepare("config-a", lora_a)`, sans l'entraîner encore, et regardez le premier exemple tel que le *trainer* l'a préparé (`trainer.train_dataset[0]`) : combien de ses tokens comptent dans la perte, et lesquels ? Les schémas des outils et leurs résultats y comptent-ils ? Mesurez ensuite la longueur des exemples : la plus courte, la plus longue, le total d'une époque. Le plus long tient-il dans `max_length` ? `trl` coupe un exemple trop long par la fin : qu'y perdrait-il ? Enfin, le nombre de paramètres entraînables selon `trl` est-il celui de `parametres["config-a"]` ?

**Documentation utile**

- [`SFTConfig`](https://huggingface.co/docs/trl/sft_trainer#trl.SFTConfig)
- [La perte sur les tours de l'assistant](https://huggingface.co/docs/trl/sft_trainer#train-on-assistant-messages-only)
- [L'appel d'outils avec `SFTTrainer`](https://huggingface.co/docs/trl/sft_trainer#tool-calling-with-sft)
- [`TrainingArguments`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)
- [`itertools.groupby`](https://docs.python.org/3/library/itertools.html#itertools.groupby)
- [`get_nb_trainable_parameters`](https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftModel.get_nb_trainable_parameters)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Dans les réglages : `assistant_only_loss=True` ; `report_to="tensorboard"` ; `logging_steps=1` ; `eval_strategy="steps"`, `eval_steps=0.1` ; `save_strategy="steps"`, `save_steps=0.1` ; `save_total_limit=2` ; `load_best_model_at_end=True`, `metric_for_best_model="eval_loss"`
- Un nombre sous 1, pour `eval_steps` ou `save_steps`, est une part des pas de l'entraînement : 0,1, un dixième ; pour recharger le meilleur checkpoint à la fin, `save_steps` doit être un multiple d'`eval_steps`
- Les réglages dans un dictionnaire, `reglages = dict(assistant_only_loss=True, ...)`, puis `SFTConfig(output_dir=f"sft/{nom}", **(REFERENCE | reglages | changements))` : `|` réunit deux dictionnaires, la clé de droite remplaçant la même clé à gauche
- `SFTTrainer(model=base, args=config, train_dataset=entrainement, eval_dataset=validation, processing_class=tokenizer, peft_config=lora)`
- Un exemple préparé a des `input_ids` et des `labels` de même longueur : `-100` dans `labels` exclut le token de la perte ; `itertools.groupby(zip(ids, labels), key=lambda p: p[1] != -100)` regroupe les tokens qui se suivent, ceux qui comptent et les autres ; `tokenizer.decode(tokens)` les remet en texte
- `trainer.train_dataset["input_ids"]` : les exemples tokenisés ; `trainer.model.get_nb_trainable_parameters()` : (entraînables, total)

```python
import itertools


def prepare(nom: str, lora: LoraConfig, **changements) -> SFTTrainer:
  reglages = dict(
    assistant_only_loss=...,
    report_to=...,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=0.1,
    save_strategy=...,
    save_steps=...,
    save_total_limit=2,
    load_best_model_at_end=...,
    metric_for_best_model=...,
  )
  config = SFTConfig(output_dir=f"sft/{nom}", **(REFERENCE | reglages | changements))
  return SFTTrainer(
    model=base,
    args=config,
    train_dataset=...,
    eval_dataset=...,
    processing_class=tokenizer,
    peft_config=lora,
  )


trainer = prepare("config-a", lora_a)
exemple = trainer.train_dataset[0]
paires = list(zip(exemple["input_ids"], exemple["labels"]))
comptes = [t for t, l in paires if ...]
print(f"{len(comptes)} tokens sur {len(paires)} comptent dans la perte :")
for compte, morceau in itertools.groupby(paires, key=lambda p: p[1] != -100):
  if compte:
    print(repr(tokenizer.decode([t for t, _ in morceau])))

longueurs = pd.Series([len(ids) for ids in trainer.train_dataset["input_ids"]])
print(f"De {longueurs.min()} à {longueurs.max()} tokens par exemple,", end=" ")
print(f"{longueurs.sum()} par époque ; max_length {REFERENCE['max_length']}")
print("Paramètres entraînables :", ..., "; comptés :", parametres["config-a"])
```

### Solution

In [ ]:
import itertools


def prepare(nom: str, lora: LoraConfig, **changements) -> SFTTrainer:
  reglages = dict(
    assistant_only_loss=True,
    report_to="tensorboard",
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=0.1,
    save_strategy="steps",
    save_steps=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
  )
  config = SFTConfig(output_dir=f"sft/{nom}", **(REFERENCE | reglages | changements))
  return SFTTrainer(
    model=base,
    args=config,
    train_dataset=entrainement,
    eval_dataset=validation,
    processing_class=tokenizer,
    peft_config=lora,
  )


trainer = prepare("config-a", lora_a)
exemple = trainer.train_dataset[0]
paires = list(zip(exemple["input_ids"], exemple["labels"]))
comptes = [t for t, l in paires if l != -100]
print(f"{len(comptes)} tokens sur {len(paires)} comptent dans la perte :")
for compte, morceau in itertools.groupby(paires, key=lambda p: p[1] != -100):
  if compte:
    print(repr(tokenizer.decode([t for t, _ in morceau])))

longueurs = pd.Series([len(ids) for ids in trainer.train_dataset["input_ids"]])
print(f"De {longueurs.min()} à {longueurs.max()} tokens par exemple,", end=" ")
print(f"{longueurs.sum()} par époque ; max_length {REFERENCE['max_length']}")
print(
  "Paramètres entraînables :",
  trainer.model.get_nb_trainable_parameters()[0],
  "; comptés :",
  parametres["config-a"],
)

# Seuls les tours de l'assistant comptent : chaque appel, dans son bloc <tool_call>,
# puis la réponse, chacun précédé du bloc de réflexion vide (<think></think>) et suivi
# de la fin du tour, un token sur dix environ. La consigne, les schémas, le message du
# client et les résultats des outils n'y comptent pas. Le gabarit d'entraînement de trl
# garde le bloc vide à chaque tour ; celui du modèle, à l'inférence, l'ôte des tours
# passés. Les exemples font de 1 500 à 2 000 tokens, dont 1 400 environ pour la
# consigne et les schémas : max_length=2048 n'en coupe aucun. Plus court, il couperait
# la fin, la réponse et le dernier appel : ce que la perte doit apprendre.

## 3. La configuration A, celle du cours

Lancez TensorBoard (la cellule ci-dessous, qui remet aussi à zéro le pic de mémoire), puis entraînez la configuration A, `trainer`, et suivez dans TensorBoard `train/loss`, `eval/loss` et `train/learning_rate` (il relit les journaux toutes les 30 secondes ; ⟳, en haut à droite, sans attendre). Écrivez `bilan(trainer, minutes)`, le résumé d'un entraînement fini, en un dictionnaire : sa durée en minutes, le pic de mémoire (`pic_go()`), les paramètres entraînables, la taille en Mo de l'adaptateur du meilleur checkpoint (son fichier `adapter_model.safetensors`), la meilleure perte de validation et son pas, la dernière, et le meilleur checkpoint. Rangez celui de la configuration A dans le dictionnaire `bilans`, sous `"config-a"`.

À quel pas la perte d'entraînement cesse-t-elle de chuter ? La perte de validation suit-elle ? Le pic de mémoire est-il proche de l'estimation de l'exercice 1 (`estimations`) ? Qu'est-ce que l'estimation n'a pas compté ?

Sur un T4, comptez une demi-heure environ par configuration (une estimation). Sans GPU, `entrainement` n'a que 2 exemples et `validation` 1 : deux pas d'un exemple chacun, pour voir la boucle tourner, un quart d'heure environ.

**Documentation utile**

- [TensorBoard dans un notebook](https://www.tensorflow.org/tensorboard/tensorboard_in_notebooks)
- [`TrainerState`](https://huggingface.co/docs/transformers/main_classes/callback#transformers.TrainerState)
- [`os.path.getsize`](https://docs.python.org/3/library/os.path.html#os.path.getsize)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir sft
libere()

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `trainer.state.log_history` est une liste de dictionnaires : ceux qui ont une clé `"eval_loss"` sont les évaluations, avec leur `"step"` ; `trainer.state.best_model_checkpoint`, le dossier du meilleur checkpoint
- `os.path.join(dossier, "adapter_model.safetensors")`, puis `os.path.getsize(...)`, sa taille en octets
- `pd.Series(bilans["config-a"])` l'affiche une ligne par clé

```python
def bilan(trainer: SFTTrainer, minutes: float) -> dict:
  etat = trainer.state
  evaluations = [h for h in etat.log_history if "eval_loss" in h]
  meilleure = min(evaluations, key=lambda h: h["eval_loss"])
  fichier = os.path.join(..., "adapter_model.safetensors")
  return {
    "minutes": round(minutes, 1),
    "pic_go": round(pic_go(), 1),
    "parametres": trainer.model.get_nb_trainable_parameters()[0],
    "adaptateur_mo": round(... / 1e6, 1),
    "meilleure_perte": round(meilleure["eval_loss"], 3),
    "pas_meilleure": ...,
    "derniere_perte": round(..., 3),
    "checkpoint": etat.best_model_checkpoint,
  }


debut = time.time()
trainer.train()
bilans = {"config-a": bilan(trainer, (time.time() - debut) / 60)}
print(pd.Series(bilans["config-a"]))
print("Estimation :", estimations["config-a"]["total"], "Go")
```

### Solution

In [ ]:
def bilan(trainer: SFTTrainer, minutes: float) -> dict:
  etat = trainer.state
  evaluations = [h for h in etat.log_history if "eval_loss" in h]
  meilleure = min(evaluations, key=lambda h: h["eval_loss"])
  fichier = os.path.join(etat.best_model_checkpoint, "adapter_model.safetensors")
  return {
    "minutes": round(minutes, 1),
    "pic_go": round(pic_go(), 1),
    "parametres": trainer.model.get_nb_trainable_parameters()[0],
    "adaptateur_mo": round(os.path.getsize(fichier) / 1e6, 1),
    "meilleure_perte": round(meilleure["eval_loss"], 3),
    "pas_meilleure": meilleure["step"],
    "derniere_perte": round(evaluations[-1]["eval_loss"], 3),
    "checkpoint": etat.best_model_checkpoint,
  }


debut = time.time()
trainer.train()
bilans = {"config-a": bilan(trainer, (time.time() - debut) / 60)}
print(pd.Series(bilans["config-a"]))
print("Estimation :", estimations["config-a"]["total"], "Go")

# La perte d'entraînement chute dès les premiers pas, le temps d'apprendre la forme
# des appels et des réponses, puis descend lentement ; la perte de validation suit tant
# que le modèle généralise. Le pic dépasse l'estimation : elle ne compte que l'entrée de
# chaque couche, pas les activations de la couche recalculée, ni celles du calcul de la
# perte, ni la mémoire que réserve PyTorch.

## 4. La configuration B, l'attention seule

La cellule ci-dessous retire l'adaptateur A du modèle de base, pour l'entraînement suivant. Entraînez la configuration B, `lora_b` : même rang, mêmes réglages, un seul changement, les modules ciblés, les quatre projections de l'attention au lieu de toutes les couches linéaires. Un seul changement à la fois : sinon, on ne saurait pas lequel fait la différence. Suivez-la dans le même TensorBoard, les deux runs cochés à gauche, rangez son bilan dans `bilans` sous `"config-b"`, et comparez les deux dans un tableau, une colonne par configuration.

Que coûte chaque configuration : sa durée, son pic de mémoire, ses paramètres entraînables, la taille de son adaptateur ? Le temps baisse-t-il autant que le nombre de paramètres, et pourquoi ? Laquelle a la meilleure perte de validation ? Une perte de validation plus basse fait-elle un meilleur agent ? L'exercice 5 le mesure.

**Documentation utile**

- [Comparer des runs dans TensorBoard](https://www.tensorflow.org/tensorboard/get_started)
- [`unload`](https://huggingface.co/docs/peft/package_reference/tuners#peft.tuners.tuners_utils.BaseTuner.unload)

In [ ]:
# La configuration A finie : son adaptateur retiré du modèle de base.
base = trainer.model.unload()
del trainer
libere()
%tensorboard --logdir sft

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `prepare("config-b", lora_b)` : le reste de la configuration ne change pas ; `bilan`, de l'exercice 3, résume l'entraînement
- Dans TensorBoard, le curseur *Smoothing* à 0 montre chaque pas tel quel ; les runs s'appellent `config-a/runs/<date>` et `config-b/runs/<date>`
- `pd.DataFrame(bilans)` : une colonne par configuration, une ligne par clé

```python
trainer = prepare(...)
debut = time.time()
trainer.train()
bilans["config-b"] = bilan(trainer, ...)
comparaison = pd.DataFrame(bilans)
print(comparaison)
```

### Solution

In [ ]:
trainer = prepare("config-b", lora_b)
debut = time.time()
trainer.train()
bilans["config-b"] = bilan(trainer, (time.time() - debut) / 60)
comparaison = pd.DataFrame(bilans)
print(comparaison)

# B entraîne près de trois fois moins de paramètres, et son adaptateur est d'autant
# plus petit ; le temps et le pic baissent bien moins : chaque pas traverse le modèle
# entier, en avant puis en arrière, quelles que soient les couches adaptées. La perte de
# validation compare les deux sur tous les tokens de l'assistant, dont la réponse
# rédigée fait la plus grande part, et la tâche ne juge que les appels : c'est
# l'exercice 5 qui tranche.

## 5. Mesurer et choisir

La cellule ci-dessous rend au GPU la mémoire des entraînements, et nomme les adaptateurs à mesurer, `a_mesurer` : le meilleur checkpoint de chaque configuration. Sans GPU, les vôtres n'ont vu que 2 exemples : on mesure à leur place celui que ce notebook a entraîné sur GPU avec Qwen3-0.6B, `reference`.

Chargez le modèle de base, `modele`, en `DTYPE`, sans quantification, et mesurez-le sur le jeu figé (`mesure(modele)`). Puis, pour chaque adaptateur, rechargez le modèle de base, ajoutez-lui l'adaptateur, fusionnez-le dans ses poids, et mesurez à nouveau. L'adaptateur, appris sur le modèle en 4 bits, s'applique au modèle en 16 bits : c'est ainsi qu'on le sert. Rassemblez les mesures dans un tableau, une colonne par modèle, avec, pour les configurations, leurs bilans de l'exercice 4 ; comptez les erreurs (`erreur`) de chaque modèle, et son taux `correct` selon le nombre d'appels attendus.

Choisissez enfin la configuration à garder, `choisie` : la tâche d'abord, le taux `correct` ; puis, à 3 points près, la moins coûteuse. Sur 139 demandes, un taux de 0,8 a un écart-type de 3 points environ : un écart plus petit ne départage pas deux modèles. Et le coût, une fois l'adaptateur fusionné, n'est plus celui de l'inférence, la même pour les deux, mais celui de l'entraînement et, servi sans fusion, la taille de l'adaptateur. Votre choix suit-il la perte de validation ? Que cette mesure étape par étape ne dit-elle pas de l'agent ?

**Documentation utile**

- [`PeftModel.from_pretrained`](https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftModel.from_pretrained)
- [`merge_and_unload`](https://huggingface.co/docs/peft/package_reference/tuners#peft.tuners.tuners_utils.BaseTuner.merge_and_unload)
- [`DataFrameGroupBy.mean`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.mean.html)

In [ ]:
# La mémoire des entraînements rendue au GPU.
del trainer, base
libere()
tokenizer.padding_side = "left"  # pour générer par lots

# Les adaptateurs à mesurer : le meilleur checkpoint de chaque configuration. Sans GPU,
# les vôtres n'ont vu que 2 exemples : on mesure celui que ce notebook a entraîné sur
# GPU avec Qwen3-0.6B.
a_mesurer = {nom: b["checkpoint"] for nom, b in bilans.items()}
if not GPU:
  a_mesurer = {"reference": "agent-sft-lora-qwen3-0.6b"}

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `AutoModelForCausalLM.from_pretrained(MODELE, dtype=DTYPE, device_map=DEVICE)` : sans `quantization_config`, en `DTYPE` ; `del modele` puis `libere()` avant d'en charger un autre
- `PeftModel.from_pretrained(modele, dossier)` ajoute l'adaptateur d'un dossier (un checkpoint en est un) ; `.merge_and_unload()` l'ajoute aux poids et renvoie un modèle ordinaire, aussi rapide que celui de base
- `mesure(modele)` renvoie `(taux, detail)` ; sur un T4, quelques minutes par modèle, sans GPU, une à trois minutes par demande
- `pd.concat([pd.DataFrame(mesures), pd.DataFrame(bilans)])` : les taux, puis les bilans, chaque modèle dans sa colonne (vide là où il n'a rien)
- `detail["erreur"].value_counts()` ; `detail.groupby("appels_attendus")["correct"].mean()`
- `min(proches, key=lambda nom: bilans[nom]["parametres"])` : la configuration la moins coûteuse parmi `proches`

```python
def charge_base():
  return AutoModelForCausalLM.from_pretrained(...)


mesures, details = {}, {}
modele = charge_base()
mesures["base"], details["base"] = mesure(modele)
for nom, dossier in a_mesurer.items():
  del modele
  libere()
  modele = PeftModel.from_pretrained(charge_base(), dossier).merge_and_unload()
  mesures[nom], details[nom] = ...
print(pd.concat([pd.DataFrame(mesures), pd.DataFrame(bilans).drop("checkpoint")]))
print(pd.DataFrame({nom: d["erreur"].value_counts() for nom, d in details.items()}))
print(pd.DataFrame({nom: d.groupby(...)["correct"].mean() for nom, d in ...}))

mesurees = [nom for nom in bilans if nom in mesures]
if mesurees:
  meilleur_taux = max(mesures[nom]["correct"] for nom in mesurees)
  proches = [nom for nom in mesurees if mesures[nom]["correct"] >= ...]
  choisie = min(proches, key=lambda nom: bilans[nom]["parametres"])
else:  # sans GPU, vos adaptateurs ne sont pas mesurés
  choisie = "config-a"
print("Configuration choisie :", choisie)
```

### Solution

In [ ]:
def charge_base():
  return AutoModelForCausalLM.from_pretrained(MODELE, dtype=DTYPE, device_map=DEVICE)


mesures, details = {}, {}
modele = charge_base()
mesures["base"], details["base"] = mesure(modele)
for nom, dossier in a_mesurer.items():
  del modele
  libere()
  modele = PeftModel.from_pretrained(charge_base(), dossier).merge_and_unload()
  mesures[nom], details[nom] = mesure(modele)
print(pd.concat([pd.DataFrame(mesures), pd.DataFrame(bilans).drop("checkpoint")]))
print(
  pd.DataFrame({nom: d["erreur"].value_counts() for nom, d in details.items()})
  .fillna(0)
  .astype(int)
)
print(
  pd.DataFrame(
    {nom: d.groupby("appels_attendus")["correct"].mean() for nom, d in details.items()}
  ).round(2)
)
# Deux sorties fausses du dernier modèle mesuré, à l'étape où il se trompe.
dernier = details[list(details)[-1]]
for ligne in dernier[dernier["erreur"] != "correct"].head(2).itertuples():
  print(f"\n{ligne.id}, {ligne.erreur} à l'étape {int(ligne.etape)} :")
  print(ligne.sorties[int(ligne.etape)][:300])

# La tâche d'abord ; à 3 points près, la configuration qui entraîne le moins de
# paramètres.
mesurees = [nom for nom in bilans if nom in mesures]
if mesurees:
  meilleur_taux = max(mesures[nom]["correct"] for nom in mesurees)
  proches = [nom for nom in mesurees if mesures[nom]["correct"] >= meilleur_taux - 0.03]
  choisie = min(proches, key=lambda nom: bilans[nom]["parametres"])
else:  # sans GPU, vos adaptateurs ne sont pas mesurés : la configuration du cours
  choisie = "config-a"
print("\nConfiguration choisie :", choisie)

# Le modèle de base répond souvent sans appeler l'outil attendu ; les adaptateurs
# corrigent d'abord ces erreurs-là, les appels manquants et le mauvais outil, que le
# jeu d'entraînement visait. La perte de validation ne désigne pas forcément la
# meilleure configuration : elle moyenne surtout les tokens de la réponse rédigée.
# Cette mesure juge chaque étape sur le bon historique : elle ne dit pas si l'agent,
# livré à ses propres appels, se rattrape d'une erreur ou s'égare, ni si sa réponse est
# juste. C'est l'évaluation du dernier chapitre.

## 6. Enregistrer, versionner, publier

Un checkpoint garde de quoi reprendre l'entraînement (l'état de l'optimiseur, du planning, du générateur aléatoire) ; l'adaptateur, lui, tient en deux fichiers : `adapter_model.safetensors` (les poids) et `adapter_config.json` (sa configuration, dont le modèle de base). Copiez-les du meilleur checkpoint de la configuration choisie (`bilans[choisie]["checkpoint"]`) dans un dossier `agent-sft-lora`, et comparez la taille du dossier à celle du checkpoint.

Écrivez-y aussi sa carte, `README.md` : de quoi savoir, dans six mois, d'où vient l'adaptateur, et refaire l'expérience. Elle commence par un en-tête YAML que lit le Hub (le modèle de base, la bibliothèque `peft`, la licence `apache-2.0`, celle de Qwen3), puis donne :

- les données : chaque fichier, son nombre de lignes et son empreinte SHA-256 (`hashlib`), qui désigne ses octets mêmes, d'où qu'il vienne ; et combien d'exemples ont servi ;
- la configuration de LoRA (celle d'`adapter_config.json`) et les réglages de l'entraînement ;
- son bilan (exercices 3 et 4) et ses mesures sur le jeu figé (exercice 5) ;
- les versions de `torch`, `transformers`, `peft` et `trl`.

Puis, si vous avez un compte Hugging Face, publiez le dossier sur le Hub, en **privé**, sous `<nom>/agent-sft-lora`, avec l'étiquette `v1.0` : le chapitre suivant chargera cet adaptateur. Un dépôt du Hub est un dépôt git : chaque envoi y est un commit, et l'étiquette désigne cette version-là, même après d'autres envois. Il faut le jeton `HF_TOKEN` des TP précédents, dans les secrets de Colab (la clé dans la barre de gauche) ; sans lui, la cellule ne publie rien. Sans GPU, l'adaptateur enregistré est le vôtre, de Qwen3-0.6B, entraîné sur 2 exemples et non mesuré : sa carte le dit.

**Documentation utile**

- [`shutil.copy`](https://docs.python.org/3/library/shutil.html#shutil.copy)
- [`hashlib.sha256`](https://docs.python.org/3/library/hashlib.html#hashlib.sha256)
- [Les cartes de modèle](https://huggingface.co/docs/hub/model-cards)
- [`HfApi.create_repo`](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api#huggingface_hub.HfApi.create_repo)
- [`HfApi.upload_folder`](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api#huggingface_hub.HfApi.upload_folder)
- [`HfApi.create_tag`](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api#huggingface_hub.HfApi.create_tag)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `shutil.copy(os.path.join(checkpoint, fichier), dossier)` pour chacun des deux fichiers, après `os.makedirs(dossier, exist_ok=True)` ; `taille_mo(dossier)` (cellule du début)
- `hashlib.sha256(octets).hexdigest()` : l'empreinte d'octets lus en mode binaire (`open(fichier, "rb")`)
- `json.load` sur `adapter_config.json` : `r`, `lora_alpha`, `lora_dropout`, `target_modules`, `base_model_name_or_path`
- Le jeton, comme aux TP précédents : `os.environ.get("HF_TOKEN")`, ou, dans Colab, `google.colab.userdata.get("HF_TOKEN")`, qui lève une exception quand le secret n'existe pas ; `HfApi(token=jeton).whoami()["name"]`, votre nom d'utilisateur
- `api.create_repo(depot, private=True, exist_ok=True)`, `api.upload_folder(folder_path=dossier, repo_id=depot)`, puis `api.create_tag(depot, tag="v1.0", exist_ok=True)`

````python
import hashlib
import shutil

import peft
import transformers
import trl
from huggingface_hub import HfApi

dossier = "agent-sft-lora"
choisi = bilans[choisie]
os.makedirs(dossier, exist_ok=True)
for fichier in ["adapter_model.safetensors", "adapter_config.json"]:
  shutil.copy(...)
print(
  f"Adaptateur : {taille_mo(dossier):.0f} Mo ;",
  f"checkpoint : {taille_mo(choisi['checkpoint']):.0f} Mo",
)


def empreinte(fichier: str) -> str:
  with open(fichier, "rb") as f:
    return hashlib.sha256(...).hexdigest()


def lignes(fichier: str) -> int:
  with open(fichier, encoding="utf-8") as f:
    return sum(1 for _ in f)


FICHIERS = ["agent-sft-train.jsonl", "agent-sft-validation.jsonl", "eval.jsonl"]
donnees = "\n".join(
  f"- `{f}` : {lignes(f)} lignes, SHA-256 `{empreinte(f)}`" for f in FICHIERS
)


jeton = os.environ.get("HF_TOKEN")
try:
  from google.colab import userdata

  jeton = userdata.get("HF_TOKEN")
except Exception:
  pass
api = HfApi(token=jeton)
nom = api.whoami()["name"] if jeton else "<nom>"

with open(os.path.join(dossier, "adapter_config.json")) as f:
  lora = json.load(f)
if choisie in mesures:
  resultats = ", ".join(f"{cle} {valeur}" for cle, valeur in mesures[choisie].items())
else:
  resultats = "non mesuré : entraîné sans GPU, sur 2 exemples"
carte = f"""---
base_model: {lora["base_model_name_or_path"]}
library_name: peft
license: apache-2.0
tags:
- lora
- qlora
- sft
- trl
- tool-calling
language:
- fr
---

# agent-sft-lora

Adaptateur LoRA de `{lora["base_model_name_or_path"]}` pour l'agent du service client de
Biclou & Cie (fictif) : ...

## Données

{donnees}

Entraîné sur {len(entrainement)} exemples de `agent-sft-train.jsonl`, validé sur ...

## Entraînement

- Configuration `{choisie}` : LoRA de rang {lora["r"]}, ...
- ...
- {choisi["minutes"]} min, pic {choisi["pic_go"]} Go ; meilleur checkpoint ...

## Mesures sur le jeu figé

{resultats}

## Versions

torch {torch.__version__}, transformers {...}, peft {...}, trl {...}
"""


with open(os.path.join(dossier, "README.md"), "w", encoding="utf-8") as f:
  f.write(carte)
print(carte)

if jeton:
  depot = f"{nom}/agent-sft-lora"
  api.create_repo(depot, private=..., exist_ok=True)
  api.upload_folder(folder_path=dossier, repo_id=depot)
  api.create_tag(depot, tag=..., exist_ok=True)
  print("Publié :", depot, "(v1.0)")
else:
  print("Pas de jeton HF_TOKEN : l'adaptateur et sa carte restent dans", dossier)
````

### Solution

In [ ]:
import hashlib
import shutil

import peft
import transformers
import trl
from huggingface_hub import HfApi

dossier = "agent-sft-lora"
choisi = bilans[choisie]
os.makedirs(dossier, exist_ok=True)
for fichier in ["adapter_model.safetensors", "adapter_config.json"]:
  shutil.copy(os.path.join(choisi["checkpoint"], fichier), dossier)
print(
  f"Adaptateur : {taille_mo(dossier):.0f} Mo ;",
  f"checkpoint : {taille_mo(choisi['checkpoint']):.0f} Mo",
)


def empreinte(fichier: str) -> str:
  with open(fichier, "rb") as f:
    return hashlib.sha256(f.read()).hexdigest()


def lignes(fichier: str) -> int:
  with open(fichier, encoding="utf-8") as f:
    return sum(1 for _ in f)


FICHIERS = ["agent-sft-train.jsonl", "agent-sft-validation.jsonl", "eval.jsonl"]
donnees = "\n".join(
  f"- `{f}` : {lignes(f)} lignes, SHA-256 `{empreinte(f)}`" for f in FICHIERS
)


jeton = os.environ.get("HF_TOKEN")
try:
  from google.colab import userdata

  jeton = userdata.get("HF_TOKEN")
except Exception:
  pass
api = HfApi(token=jeton)
nom = api.whoami()["name"] if jeton else "<nom>"

with open(os.path.join(dossier, "adapter_config.json")) as f:
  lora = json.load(f)
if choisie in mesures:
  resultats = f"Sur {len(eval_part)} demandes, étape par étape : " + ", ".join(
    f"{cle} {valeur}" for cle, valeur in mesures[choisie].items()
  )
  resultats += f" ; modèle de base : correct {mesures['base']['correct']}"
else:
  resultats = "non mesuré : entraîné sans GPU, sur 2 exemples"
carte = f"""---
base_model: {lora["base_model_name_or_path"]}
library_name: peft
license: apache-2.0
tags:
- lora
- qlora
- sft
- trl
- tool-calling
language:
- fr
---

# agent-sft-lora

Adaptateur LoRA de `{lora["base_model_name_or_path"]}` pour l'agent du service client de
Biclou & Cie (fictif) : avec la consigne de l'agent et les schémas de ses cinq outils,
il appelle les outils utiles, un à la fois, puis répond au client selon la charte.

## Données

{donnees}

Entraîné sur {len(entrainement)} exemples de `agent-sft-train.jsonl`, validé sur
{len(validation)} de `agent-sft-validation.jsonl`, mesuré sur `eval.jsonl`, le jeu
d'évaluation figé.

## Entraînement

- Configuration `{choisie}` : LoRA de rang {lora["r"]}, `lora_alpha` {lora["lora_alpha"]},
  dropout {lora["lora_dropout"]}, sur {", ".join(sorted(lora["target_modules"]))}
- Modèle de base {"en NF4, double quantification" if GPU else "en float32"}
- `SFTTrainer`, perte sur les tours de l'assistant : {REFERENCE["num_train_epochs"]}
  époque(s), taux {REFERENCE["learning_rate"]} en cosinus après
  {REFERENCE["warmup_steps"]:.0%} de montée, `{REFERENCE["optim"]}`, {LOT * ACCUMULATION}
  exemple(s) par mise à jour, `max_length` {REFERENCE["max_length"]}
- {choisi["minutes"]} min, pic {choisi["pic_go"]} Go ; meilleur checkpoint
  `{os.path.basename(choisi["checkpoint"])}`, perte de validation
  {choisi["meilleure_perte"]}

## Mesures sur le jeu figé

{resultats}

## Versions

torch {torch.__version__}, transformers {transformers.__version__}, peft
{peft.__version__}, trl {trl.__version__}
"""


with open(os.path.join(dossier, "README.md"), "w", encoding="utf-8") as f:
  f.write(carte)
print(carte)

if jeton:
  depot = f"{nom}/agent-sft-lora"
  api.create_repo(depot, private=True, exist_ok=True)
  api.upload_folder(folder_path=dossier, repo_id=depot)
  api.create_tag(depot, tag="v1.0", exist_ok=True)
  print("Publié :", depot, "(v1.0)")
else:
  print("Pas de jeton HF_TOKEN : l'adaptateur et sa carte restent dans", dossier)

# Le dossier pèse quelques dizaines de Mo, le checkpoint bien plus : il garde aussi les
# états de l'optimiseur. Les empreintes désignent les données exactes : un autre jeu,
# même du même nom, aurait d'autres empreintes. L'étiquette v1.0 reste sur cette
# version : un nouvel envoi fait un nouveau commit, sans la déplacer.

## À retenir

- Avant d'entraîner, on estime la mémoire : les poids gelés en 4 bits, l'adaptateur avec ses gradients et les états de l'optimiseur, presque rien, et les activations, que le *gradient checkpointing* réduit à l'entrée de chaque couche. L'affinage complet du même modèle ne tiendrait pas sur le GPU.
- `assistant_only_loss` limite la perte aux tours de l'assistant : ses appels et sa réponse, pas la consigne, les schémas ni les résultats des outils, qui font l'essentiel de chaque exemple.
- On compare deux configurations en ne changeant qu'un réglage. Adapter moins de couches entraîne moins de paramètres, mais ne raccourcit guère l'entraînement : chaque pas traverse le modèle entier.
- On choisit sur la tâche, mesurée sur le jeu figé, pas sur la perte de validation, puis, à résultats proches, sur le coût. La mesure étape par étape juge chaque appel sur le bon historique ; l'agent de bout en bout reste à évaluer.
- Un adaptateur se refait et se retrouve avec sa carte : le modèle de base, les empreintes des données, la configuration, les réglages, les mesures et les versions ; publié et étiqueté, c'est la version que chargera l'agent.